# AI Applications — AI Product Description Generator

**Project objective:** I built a Flask web application in Google Colab that uses a small Hugging Face language model to create product descriptions based on a product name, user-provided features, and a selected writing tone.

**LLM approach:** Option B — Hugging Face Transformers `pipeline("text-generation")` with Qwen2.5-Instruct. The public model runs in Colab, so I do not need an OpenAI API key.

**Application features:** Input validation, four writing tones, an LLM-powered description generator, a basic unsupported-claim screen, one automatic rewrite attempt, and a transparent fact-only fallback. The final description still requires human review.

## Step 1 — Install the libraries

Use Flask for the web interface and the Hugging Face Transformers `pipeline` class to load the Qwen language model. PyTorch provides model execution on the Colab CPU or GPU. If a GPU is available, the notebook uses the larger 1.5B model. Otherwise, it uses the smaller 0.5B model.

Google Colab already has a Python environment, so I do not create a separate virtual environment or project folder.

In [1]:
# Install the libraries for this application.
%pip -q install "flask>=3.0,<4" "transformers>=4.55,<5" "accelerate>=1.0" "safetensors>=0.4"

import torch
print("Device:", "GPU (CUDA)" if torch.cuda.is_available() else "CPU")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
Device: GPU (CUDA)


## Step 2 — Prepare Flask's template location

Flask expects HTML templates in a directory named `templates`. I create only that required directory in Colab's working location. The Python program will be saved directly as `app.py`.

In [2]:
# Create the one folder Flask needs for its HTML template.
%cd /content
from pathlib import Path
Path("templates").mkdir(exist_ok=True)
print("Template folder is ready.")

/content
Template folder is ready.


## Step 3 — Write the Python application (`app.py`)

The application uses the Hugging Face `pipeline("text-generation")` class with a Qwen instruction-tuned language model to create original descriptions in four tones. I separated the code into functions for reading keywords, building prompts, generating drafts, screening common unsupported claims, and processing Flask requests.

A prompt alone does not guarantee factual accuracy. For this reason, the code checks drafts for a selected set of risky marketing claims and unlisted numbers. If it detects an issue, the AI gets one chance to revise the text. If the new draft still triggers a warning, the application displays a clearly labeled, conservative description built from the submitted features. **This screening is not complete fact verification**, so every result must be reviewed before publication.

In [3]:
%%writefile app.py
"""AI Product Description Generator: LLM writing with a basic claim-screening safeguard."""

import re
import threading

import torch
from flask import Flask, render_template, request

app = Flask(__name__)

# These are writing instructions. A tone must never change the actual facts.
TONE_GUIDES = {
    "friendly": (
        "Use a warm, welcoming, conversational voice. Use simple, helpful language. "
        "Avoid exaggeration and keep the writing approachable."
    ),
    "professional": (
        "Use a factual, concise, polished business voice. Focus on clear product details. "
        "Avoid exclamation marks, hype, or personal stories."
    ),
    "fun": (
        "Use an upbeat, playful voice with a little personality. "
        "A light exclamation is fine, but do not invent product benefits."
    ),
    "luxury": (
        "Use an elegant, understated editorial voice, as in a modern design catalogue. "
        "Use graceful phrasing and a calm rhythm, not empty luxury buzzwords. "
        "Do not say premium, crafted, exclusive, finest, exceptional quality, "
        "or superior unless these specific details are supplied. No exclamation marks."
    ),
}

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = (
    "Qwen/Qwen2.5-1.5B-Instruct" if DEVICE == "cuda"
    else "Qwen/Qwen2.5-0.5B-Instruct"
)
_generator = None
_generator_lock = threading.Lock()


def get_generator():
    """Load the Hugging Face text-generation pipeline once when needed."""
    global _generator
    with _generator_lock:
        if _generator is None:
            # Option B uses the pipeline class requested in the assignment.
            # A local Colab model does not need a paid API key.
            from transformers import pipeline
            print(f"Loading {MODEL_NAME} with the Transformers pipeline on {DEVICE}...")
            _generator = pipeline(
                task="text-generation",
                model=MODEL_NAME,
                device=0 if DEVICE == "cuda" else -1,
                dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
            )
    return _generator


def parse_keywords(raw):
    """Split, trim and de-duplicate comma-separated user-supplied facts."""
    features = []
    for item in raw.split(","):
        value = " ".join(item.strip().split())
        if value and value.casefold() not in [f.casefold() for f in features]:
            features.append(value)
    return features


def make_messages(product, features, tone, feedback=""):
    """Keep the LLM prompt separate from the user interface and claim checks."""
    rules = (
        "You are an e-commerce copywriter. Only the product name and feature list "
        "are supplied product facts. Write an engaging 2-3 sentence product description "
        "for an online store. Describe the provided features accurately. "
        "Never add unlisted materials, certifications, specifications, durations, "
        "performance results, warranties, quality rankings, or extra use cases. "
        "Do not write 'premium materials', 'perfect fit', 'unmatched sound quality', "
        "or 'advanced technology' without evidence. "
        "If details are sparse, keep the description short. "
        "Output only the description, no title, markdown or explanations."
    )
    user_prompt = (
        f"Product: {product}\nUser-supplied features: {', '.join(features)}\n"
        f"Tone instruction: {TONE_GUIDES[tone]}\n"
        "Keep all factual statements grounded in the exact features above."
    )
    if feedback:
        user_prompt += (
            "\nThe last draft contained these potentially unsupported claims: "
            + feedback + ". Remove them. Do not replace them with new claims."
        )
    return [{"role": "system", "content": rules},
            {"role": "user", "content": user_prompt}]


def generate_draft(product, features, tone, feedback=""):
    """Generate a Qwen product description with Hugging Face's pipeline."""
    generator = get_generator()
    messages = make_messages(product, features, tone, feedback)

    # The text-generation pipeline formats chat messages with Qwen's chat template.
    # return_full_text=False keeps the prompt out of the displayed description.
    # The lock protects a shared generator from simultaneous Flask requests.
    with _generator_lock:
        with torch.inference_mode():
            result = generator(
                messages,
                max_new_tokens=125,
                do_sample=False,
                repetition_penalty=1.12,
                return_full_text=False,
                pad_token_id=generator.tokenizer.eos_token_id,
            )

    draft = result[0]["generated_text"]
    # Handle chat outputs that include an assistant message dictionary.
    if isinstance(draft, list):
        draft = draft[-1]["content"]
    draft = draft.strip()
    if not draft:
        raise RuntimeError("The language model returned an empty description.")
    return draft


# COMMON claim patterns are a screening list, NOT a comprehensive fact-checker.
# A claim is allowed if that same wording appears in the user's feature list.
RISKY_CLAIMS = {
    "premium materials": r"\bpremium (?:quality )?materials?\b",
    "superior or unmatched quality": r"\b(?:unmatched|unparalleled|superior|best-in-class|top-of-the-line)\b",
    "unverified technology level": r"\b(?:advanced|cutting-edge|state-of-the-art) (?:technology|noise cancellation|features?)\b",
    "perfect fit": r"\bperfect fit\b|\bfits? perfectly\b",
    "sound-quality promise": r"\b(?:crystal.clear|studio.quality|unmatched|exceptional) sound\b|\bsound clarity\b",
    "battery or charging claim": r"\b(?:battery life|fast charging|quick charge|rechargeable battery)\b",
    "material claim": r"\b(?:stainless steel|genuine leather|organic cotton|BPA.free|aluminum alloy)\b",
    "water resistance": r"\b(?:waterproof|water.resistant|sweatproof)\b",
    "certification or warranty": r"\b(?:certified|FDA.approved|lifetime warranty|money.back guarantee)\b",
    "unsupported use case": r"\b(?:intense workouts?|long flights?|underwater use|extreme sports)\b",
    "absolute promise": r"\b(?:guaranteed|flawless|100%|zero distractions|complete silence)\b",
}


def screen_claims(description, product, features):
    """Flag some common invented claims and numbers not supplied by the user.

    This is deliberately conservative but incomplete; passing does NOT mean
    an AI-generated description is factually verified.
    """
    source_text = " ".join([product] + features).casefold()
    findings = []
    for label, pattern in RISKY_CLAIMS.items():
        for match in re.finditer(pattern, description, flags=re.I):
            if match.group(0).casefold() not in source_text:
                findings.append(label)
                break

    # Specific numbers (e.g., 24-hour battery life) must appear in the input.
    supplied_numbers = set(re.findall(r"\b\d+(?:\.\d+)?\b", source_text))
    output_numbers = set(re.findall(r"\b\d+(?:\.\d+)?\b", description))
    if output_numbers - supplied_numbers:
        findings.append("new numbers or measurements")

    if len(description) > 850:
        findings.append("description is too long")
    return list(dict.fromkeys(findings))


def join_features(features):
    """Join the exact supplied words without claiming new specifications."""
    if len(features) == 1:
        return features[0]
    if len(features) == 2:
        return " and ".join(features)
    return ", ".join(features[:-1]) + ", and " + features[-1]


def grounded_fallback(product, features, tone):
    """Create conservative copy only from the input when both LLM drafts are flagged.

    This template is a transparent fallback, not a claim that the LLM was verified.
    """
    feature_text = join_features(features)
    templates = {
        "friendly": (
            f"Meet {product}! Its listed features include {feature_text}. "
            "A simple way to see the details that matter to you."
        ),
        "professional": (
            f"{product} is listed with the following features: {feature_text}. "
            "This description focuses on the supplied specifications without added claims."
        ),
        "fun": (
            f"Say hello to {product}! On the feature list: {feature_text}. "
            "That's a good place to start exploring!"
        ),
        "luxury": (
            f"A considered introduction to {product}. Its defining listed details are "
            f"{feature_text}. An understated presentation of the essentials."
        ),
    }
    return templates[tone]


def generate_description(product, features, tone):
    """Generate, screen, retry once, then use a grounded fallback if needed.

    Returns a result dictionary so the page can explain what happened.
    """
    if not product.strip() or not features or tone not in TONE_GUIDES:
        raise ValueError("A product, features and valid tone are required.")

    all_findings = []
    feedback = ""
    for attempt in range(2):
        draft = generate_draft(product, features, tone, feedback=feedback)
        findings = screen_claims(draft, product, features)
        if not findings:
            return {
                "description": draft,
                "mode": "revised" if attempt else "generated",
                "findings": all_findings,
            }
        all_findings = list(dict.fromkeys(all_findings + findings))
        feedback = ", ".join(findings)

    return {
        "description": grounded_fallback(product, features, tone),
        "mode": "fallback",
        "findings": all_findings,
    }


@app.route("/", methods=["GET", "POST"])
def home():
    """Show the input form and generate the description after submission."""
    result = None
    error = None
    product = ""
    raw_keywords = ""
    tone = "friendly"

    if request.method == "POST":
        product = " ".join(request.form.get("product_name", "").split())
        raw_keywords = request.form.get("keywords", "").strip()
        tone = request.form.get("tone", "friendly")
        features = parse_keywords(raw_keywords)

        # Server-side validation is important even when HTML inputs are required.
        if not product or not features:
            error = "Enter a product name and at least one real product feature."
        elif len(product) > 100 or len(raw_keywords) > 250:
            error = "The product name or feature list is too long."
        elif len(features) > 10 or any(len(f) > 70 for f in features):
            error = "Use at most 10 short, comma-separated features."
        elif tone not in TONE_GUIDES:
            error = "Choose a valid writing tone."
        else:
            try:
                result = generate_description(product, features, tone)
            except Exception:
                app.logger.exception("Text generation failed")
                error = "The model could not generate text. Check Colab output and try again."

    return render_template(
        "index.html",
        result=result,
        error=error,
        product_name=product,
        keywords=raw_keywords,
        tone=tone,
        tone_options=TONE_GUIDES,
    )


if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000, debug=False, use_reloader=False)

Writing app.py


## Step 4 — Create the web interface (`templates/index.html`)

The HTML page contains a product-name field, comma-separated feature input, writing tone menu, and output section. A message explains whether the text comes directly from the LLM, from a revised LLM draft, or from the fact-only fallback. I added a copy button, clear input guidance, and responsive styling so the interface is easy to use.

Flask's template engine escapes displayed text, and the Python route validates inputs before generating descriptions.

In [4]:
%%writefile templates/index.html
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>AI Product Description Generator</title>
  <style>
    /* The interface stays readable on a laptop or a mobile screen. */
    * { box-sizing: border-box; }
    body { margin:0; padding:35px 14px; font-family:Arial, Helvetica, sans-serif;
           background:#f4f7fc; color:#192842; }
    .app { max-width:720px; margin:auto; padding:30px; border-radius:18px;
           border:1px solid #e0e7f0; background:#fff;
           box-shadow:0 10px 35px rgba(26,47,95,.06); }
    h1 { margin:0 0 8px; font-size:27px; letter-spacing:-.6px; }
    .intro { font-size:14px; color:#596987; line-height:1.65; margin:0 0 24px; }
    label { display:block; font-weight:700; font-size:14px; margin:17px 0 7px; }
    input, select { display:block; width:100%; padding:12px 14px; font:inherit;
                    color:#192842; border:1px solid #c9d4e5; border-radius:9px;
                    background:#fff; }
    input:focus, select:focus, button:focus { outline:3px solid #b7ceff; outline-offset:1px; }
    .hint { font-size:12px; color:#71809a; margin:6px 0 0; }
    button { border:0; border-radius:9px; cursor:pointer; font:inherit; font-weight:700; }
    .generate { width:100%; background:#345dd8; color:white; padding:13px 16px; margin-top:24px; }
    .generate:hover { background:#264ab7; }
    .result { margin-top:26px; border:1px solid #c9d9ff; background:#f0f5ff;
              border-radius:12px; padding:19px; }
    .result h2 { margin:0 0 12px; font-size:18px; }
    .result p { margin:12px 0 0; line-height:1.75; font-size:15px; white-space:pre-wrap; }
    .status { font-size:12px; color:#425777; margin:0; }
    .warning { margin-top:15px; padding:12px 14px; border-radius:9px;
               background:#fff9ed; border:1px solid #efd69b; font-size:13px;
               line-height:1.55; }
    .error { margin-top:20px; border:1px solid #f2b5b5; border-radius:9px;
             background:#fff3f3; padding:13px; color:#992d2d; }
    .actions { display:flex; justify-content:flex-end; margin-top:14px; }
    .copy { background:white; color:#345dd8; border:1px solid #b5c7f5; padding:9px 14px; }
    footer { margin-top:24px; font-size:12px; text-align:center; color:#79869c; line-height:1.5; }
    @media(max-width:540px) { .app {padding:20px;} h1 {font-size:23px;} }
  </style>
</head>
<body>
  <main class="app">
    <h1>AI Product Description Generator</h1>
    <p class="intro">Create product copy from features you provide. Choose a writing tone,
      then review the description before using it in a store.</p>

    <!-- Flask receives the POST request and handles input validation. -->
    <form method="post">
      <label for="product_name">Product name</label>
      <input id="product_name" name="product_name" maxlength="100" required
             placeholder="e.g., Wireless Bluetooth Headphones"
             value="{{ product_name }}">

      <label for="keywords">Product features (separated by commas)</label>
      <input id="keywords" name="keywords" maxlength="250" required
             placeholder="e.g., lightweight, wireless, noise cancellation"
             value="{{ keywords }}">
      <p class="hint">Enter only details you know are true. Keywords are treated as the source of product facts.</p>

      <!-- Optional stretch goal: four clearly different customer-facing styles. -->
      <label for="tone">Writing tone</label>
      <select id="tone" name="tone">
        {% for key, guide in tone_options.items() %}
          <option value="{{ key }}" {% if tone == key %}selected{% endif %}>{{ key | title }}</option>
        {% endfor %}
      </select>
      <button type="submit" class="generate">Generate description</button>
    </form>

    {% if error %}<div class="error" role="alert">{{ error }}</div>{% endif %}

    {% if result %}
      <section class="result" aria-live="polite">
        <h2>Product description</h2>
        <!-- Explain whether the AI draft or the conservative fallback is shown. -->
        {% if result.mode == 'generated' %}
          <div class="status">AI-generated draft · Passed basic automated screening</div>
        {% elif result.mode == 'revised' %}
          <div class="status">AI-generated revision · Passed basic automated screening</div>
        {% else %}
          <div class="status">Fact-only fallback · An AI draft was screened out</div>
        {% endif %}
        <!-- Jinja escapes model output, so untrusted text is not interpreted as HTML. -->
        <p id="description-text">{{ result.description }}</p>
        {% if result.mode == 'fallback' %}
          <div class="warning">
            <strong>Why the fallback appeared:</strong> The AI drafts contained possible
            unsupported claims ({{ result.findings | join(', ') }}).
            This simpler version uses only the submitted feature list.
          </div>
        {% endif %}
        <div class="actions"><button type="button" class="copy" onclick="copyDescription()">Copy description</button></div>
      </section>
    {% endif %}

    <footer>Automated screening catches some risky claims, not all of them.
      Check every description against the real product before publishing.</footer>
  </main>

  <script>
    // Optional convenience feature: copy the displayed text to the clipboard.
    async function copyDescription() {
      const text = document.getElementById('description-text').innerText;
      try {
        await navigator.clipboard.writeText(text);
        document.querySelector('.copy').textContent = 'Copied!';
      } catch (error) {
        // Clipboard permission may be restricted inside a Colab iframe.
        const selection = window.getSelection();
        const range = document.createRange();
        range.selectNodeContents(document.getElementById('description-text'));
        selection.removeAllRanges();
        selection.addRange(range);
        document.querySelector('.copy').textContent = 'Selected — press Ctrl+C';
      }
    }
  </script>
</body>
</html>

Writing templates/index.html


## Step 5 — Test the claim-screening logic without loading the model

I first test individual Python functions. These checks look for marketing claims that were not included in the user-provided features, confirm that supplied claims are not automatically rejected, and check that a conservative fallback contains the original features. This keeps basic debugging separate from the slower language model generation.

In [5]:
# Import the Python file and test the claim-screening functions.
from app import screen_claims, grounded_fallback, parse_keywords, TONE_GUIDES

features = parse_keywords("wireless, lightweight, noise cancellation, comfortable ear cushions")
product = "Wireless Bluetooth Headphones"

# This is an example of a draft that should trigger screening.
unsafe_draft = "Premium materials provide unmatched sound quality and 24-hour battery life."
problems = screen_claims(unsafe_draft, product, features)
print("Flagged issues:", problems)
assert "premium materials" in problems
assert "new numbers or measurements" in problems

# A description limited to supplied facts should pass these particular rules.
source_draft = "Wireless Bluetooth Headphones with lightweight design and comfortable ear cushions."
assert not screen_claims(source_draft, product, features)

# Confirm that every tone has a fallback using the entered features.
for tone_name in TONE_GUIDES:
    fallback = grounded_fallback(product, features, tone_name)
    assert all(feature in fallback for feature in features)
    print(f"{tone_name.title()}: {fallback}")

print("Basic screening and tone tests passed.")

Flagged issues: ['premium materials', 'superior or unmatched quality', 'sound-quality promise', 'battery or charging claim', 'new numbers or measurements']
Friendly: Meet Wireless Bluetooth Headphones! Its listed features include wireless, lightweight, noise cancellation, and comfortable ear cushions. A simple way to see the details that matter to you.
Professional: Wireless Bluetooth Headphones is listed with the following features: wireless, lightweight, noise cancellation, and comfortable ear cushions. This description focuses on the supplied specifications without added claims.
Fun: Say hello to Wireless Bluetooth Headphones! On the feature list: wireless, lightweight, noise cancellation, and comfortable ear cushions. That's a good place to start exploring!
Luxury: A considered introduction to Wireless Bluetooth Headphones. Its defining listed details are wireless, lightweight, noise cancellation, and comfortable ear cushions. An understated presentation of the essentials.
Basic sc

## Step 6 — Test the Flask form and automatic fallback

I use Flask's test client to check the webpage without starting a live server. These tests cover missing inputs, invalid tone values, HTML escaping, the pipeline call, and the important case where the LLM keeps producing unsupported claims.

For the last test, I temporarily replace the LLM generation function with sample text. This lets me check the fallback behavior without relying on a particular random model response. It does **not** replace the real model during normal use.

In [6]:
# Automated functional checks with sample generated text.
from unittest.mock import patch
from app import app as flask_app, generate_description

client = flask_app.test_client()
assert client.get("/").status_code == 200

empty = client.post("/", data={"product_name": "", "keywords": ""})
assert b"Enter a product name" in empty.data

bad_tone = client.post("/", data={
    "product_name": "Headphones", "keywords": "lightweight", "tone": "unknown"
})
assert b"Choose a valid writing tone" in bad_tone.data

# The model is mocked only for this test: both drafts contain fabricated claims.
with patch("app.generate_draft", return_value="Premium materials and unmatched quality, 24 hours!") as mock_llm:
    checked = generate_description(product, features, "luxury")
    assert checked["mode"] == "fallback"
    assert "premium materials" not in checked["description"].lower()
    assert mock_llm.call_count == 2

# Check that text is escaped rather than interpreted as HTML.
with patch("app.generate_description", return_value={
    "description": "Safe <script>alert(1)</script>", "mode": "generated", "findings": []
}):
    response = client.post("/", data={
        "product_name": "Sample product", "keywords": "lightweight", "tone": "friendly"
    })
    assert b"&lt;script&gt;" in response.data

# Check that the pipeline receives chat messages and returns only generated text.
# This mock tests the pipeline call without downloading the model again.
from unittest.mock import MagicMock
from app import generate_draft

fake_pipeline = MagicMock(return_value=[{
    "generated_text": "A lightweight pair of wireless headphones."
}])
fake_pipeline.tokenizer.eos_token_id = 151645
with patch("app.get_generator", return_value=fake_pipeline):
    draft = generate_draft(product, features, "professional")
    assert draft == "A lightweight pair of wireless headphones."
    args, kwargs = fake_pipeline.call_args
    assert args[0][0]["role"] == "system"
    assert args[0][1]["role"] == "user"
    assert kwargs["return_full_text"] is False

print("Flask, fallback, HTML escaping, and pipeline integration tests passed.")

Flask, fallback, HTML escaping, and pipeline integration tests passed.


## Step 7 — Load the language model and generate a real description

The earlier tests used sample text, but this step runs an actual Qwen model through the Hugging Face `pipeline("text-generation")` interface. The model is downloaded the first time this cell runs. On a CPU, generation may be slower than on a GPU.

The output includes a mode (`generated`, `revised`, or `fallback`). The fallback is used only when the AI drafts trigger the defined screening rules. It is not a substitute for human fact-checking.

In [7]:
# Generate one real description with the downloaded Qwen model.
from app import MODEL_NAME, DEVICE, generate_description

print("Model:", MODEL_NAME)
print("Device:", DEVICE)
result = generate_description(
    "Wireless Bluetooth Headphones",
    ["wireless", "lightweight", "noise cancellation", "comfortable ear cushions"],
    "luxury",
)
print("Output mode:", result["mode"])
print("Description:", result["description"])
if result["findings"]:
    print("Issues detected in earlier drafts:", result["findings"])

Model: Qwen/Qwen2.5-1.5B-Instruct
Device: cuda
Loading Qwen/Qwen2.5-1.5B-Instruct with the Transformers pipeline on cuda...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Output mode: fallback
Description: A considered introduction to Wireless Bluetooth Headphones. Its defining listed details are wireless, lightweight, noise cancellation, and comfortable ear cushions. An understated presentation of the essentials.
Issues detected in earlier drafts: ['superior or unmatched quality', 'unverified technology level']


## Step 8 — Open the live Flask application inside Colab

I start Flask in a background thread and display it using Colab's built-in port preview. The addresses printed by Flask (such as `127.0.0.1:5000`) refer to the Colab runtime, so they are not normal links to open on my own computer.

**Important:** If I change the Python file after launching Flask, I restart the Colab runtime and run the notebook again. Otherwise, the existing server may continue using old code.

In [8]:
# Run Flask in a background thread and show the Colab preview.
import threading
import time
from google.colab import output
from app import app as flask_app

def run_server():
    flask_app.run(host="0.0.0.0", port=5000, debug=False, use_reloader=False)

# Start one Flask server per Colab runtime.
if not any(t.name == "product-description-flask" and t.is_alive()
           for t in threading.enumerate()):
    thread = threading.Thread(target=run_server, name="product-description-flask", daemon=True)
    thread.start()
    time.sleep(2)

print("Use the application below. The printed local server addresses are not public links.")
output.serve_kernel_port_as_iframe(5000, height=890)

 * Serving Flask app 'app'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


Use the application below. The printed local server addresses are not public links.


<IPython.core.display.Javascript object>

## Step 9 — Test different products and writing tones

I can compare descriptions by using the same product details with different tones. This helps me see whether changing the tone affects the writing style without changing the underlying product facts.

| Product | Features | Suggested tones |
|---|---|---|
| Wireless Bluetooth Headphones | wireless, lightweight, noise cancellation, comfortable ear cushions | Friendly, Professional, Fun, Luxury |
| Reusable Water Bottle | reusable, stainless steel, 750 mL, leakproof lid | Professional, Luxury |
| Canvas Tote Bag | canvas fabric, zipper closure, two handles | Fun, Friendly |
